# Monthly Retail CDAO Run

Run this notebook once a month. It does everything end-to-end:

1. **SFTP** your monthly CSV from `avalonwinscp.fnb.co.za`
2. **Hash** the PII columns (`cust_id_reg_no`, `EMAIL_ADDR`, `CUST_CELL_NO`)
3. **Convert** CSV → Parquet (chunked, memory-safe)
4. **Upload** the Parquet to `gs://customer_spend_data/`

## How to use

1. Set the **`STAMP`** in Cell 2 to the date you want (defaults to today).
2. Leave **`USE_TEST_BUCKET = True`** for the first run — uploads to your test bucket, prod is untouched.
3. Run all cells top to bottom (`Shift + Enter` through each, or **Run All**).
4. Once happy with the test run, flip `USE_TEST_BUCKET = False` and re-run the upload cell.

## Prerequisites

- VPN connected
- `gcloud auth application-default login` done once on this machine
- `.env` file in this folder with `AD_USERNAME`, `AD_PASSWORD`, `TEST_BUCKET` (copy `.env.example` if you haven't)

## Cell 1 — Install / check packages

Safe to re-run. Restart the kernel after this cell if anything was newly installed.

In [ ]:
%pip install --quiet paramiko pandas pyarrow google-cloud-storage python-dotenv
print('\n' + '─' * 60)
print('✓ Packages installed.')
print('─' * 60)
print('\n→ If anything was newly installed: Kernel → Restart, then continue with Cell 2.')
print('  If they were all already there, just go to Cell 2.')

## Cell 2 — Settings

This is the **only cell you normally edit**. Change `STAMP` and the bucket choice; everything else stays the same.

In [ ]:
from datetime import date

# ── What to process ──────────────────────────────────────────────
STAMP = date.today().strftime('%Y%m%d')    # e.g. '20260520' — change if you want a specific date
STEM  = 'burger'                            # file stem; full name = <STEM>_<STAMP>.csv

# ── Where to upload ──────────────────────────────────────────────
USE_TEST_BUCKET = True       # ← TRUE for safe test runs; flip to False for the real prod upload

# ── SFTP source ──────────────────────────────────────────────────
SFTP_HOST  = 'avalonwinscp.fnb.co.za'
SFTP_PORT  = 22
REMOTE_DIR = '/data/fnb/retail_sales_and_cdao/Pierre/'

# ── GCP ──────────────────────────────────────────────────────────
GCP_PROJECT = 'fmn-sandbox'

# ── Local cache ──────────────────────────────────────────────────
OUT_DIR = './out'

# ── Processing knobs ─────────────────────────────────────────────
CHUNKSIZE     = 500_000          # rows per CSV chunk — flat memory
COMPRESSION   = 'snappy'         # parquet compression
HASH_COLUMNS  = ['cust_id_reg_no', 'EMAIL_ADDR', 'CUST_CELL_NO']
INT_COLS      = ['demo_1', 'demo_4', 'demo_8']
FLOAT_COLS    = ['demo_2', 'demo_3', 'demo_7']

print(f'  STAMP:         {STAMP}')
print(f'  Source file:   {STEM}_{STAMP}.csv')
print(f'  Bucket mode:   {"TEST" if USE_TEST_BUCKET else "PROD"}')
print(f'  GCP project:   {GCP_PROJECT}')

## Cell 3 — Load credentials + resolve bucket

Reads `AD_USERNAME`, `AD_PASSWORD`, `GCP_BUCKET`, `TEST_BUCKET` from your `.env` file (or env vars). Prompts you for anything missing.

In [ ]:
import os, getpass
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# AD credentials
AD_USERNAME = os.getenv('AD_USERNAME', '').strip()
AD_PASSWORD = os.getenv('AD_PASSWORD') or os.getenv('DOMAIN_PW') or ''

if not AD_USERNAME:
    AD_USERNAME = input('AD username (e.g. f3799182): ').strip()
if not AD_PASSWORD:
    AD_PASSWORD = getpass.getpass('AD password (hidden as you type): ')

# Bucket
PROD_BUCKET = os.getenv('GCP_BUCKET', 'customer_spend_data')
TEST_BUCKET = os.getenv('TEST_BUCKET', '')

if USE_TEST_BUCKET:
    if not TEST_BUCKET:
        raise RuntimeError(
            'USE_TEST_BUCKET is True but TEST_BUCKET is not set in .env. '
            'Add TEST_BUCKET=your-test-bucket-name to retail_cdao/.env'
        )
    BUCKET = TEST_BUCKET
    BUCKET_LABEL = f'TEST: {BUCKET}'
else:
    BUCKET = PROD_BUCKET
    BUCKET_LABEL = f'PROD: {BUCKET}  ⚠ this is the real bucket'

print(f'✓ Credentials captured for: {AD_USERNAME}')
print(f'✓ Will upload to: {BUCKET_LABEL}')

## Cell 4 — Preview

Last chance to sanity-check before we touch anything.

In [ ]:
file_basename = f'{STEM}_{STAMP}'
remote_path   = REMOTE_DIR.rstrip('/') + '/' + file_basename + '.csv'
out_dir       = Path(OUT_DIR).expanduser()
local_csv     = out_dir / f'{file_basename}.csv'
local_parq    = out_dir / f'{file_basename}.parquet'

print('=' * 60)
print('  About to run:')
print('=' * 60)
print(f'  Source CSV:   {remote_path}')
print(f'  Local CSV:    {local_csv}')
print(f'  Local Parq:   {local_parq}')
print(f'  Hash columns: {HASH_COLUMNS}')
print(f'  Upload to:    gs://{BUCKET}/{file_basename}.parquet')
print('=' * 60)
print('\n→ If anything above looks wrong, edit Cell 2 and re-run.')
print('→ Otherwise, continue with Cell 5.')

## Cell 5 — SFTP the CSV from avalonwinscp

In [ ]:
import time, paramiko

out_dir.mkdir(parents=True, exist_ok=True)

print(f'Connecting to {SFTP_HOST}:{SFTP_PORT} as {AD_USERNAME}...')
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(
    hostname=SFTP_HOST, port=SFTP_PORT,
    username=AD_USERNAME, password=AD_PASSWORD,
    timeout=30, allow_agent=False, look_for_keys=False,
)
sftp = ssh.open_sftp()

try:
    print(f'Downloading {remote_path} ...')
    t0 = time.time()
    sftp.get(remote_path, str(local_csv))
    size_mb = local_csv.stat().st_size / 1e6
    print(f'✓ {size_mb:.1f} MB in {time.time()-t0:.1f}s → {local_csv}')
finally:
    sftp.close()
    ssh.close()

## Cell 6 — Convert CSV → Parquet (with PII hashing)

In [ ]:
import hashlib, time
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

def hash_value(val):
    if pd.isna(val):
        return val
    s = str(val).lower().replace(' ', '')
    return hashlib.sha256(s.encode('utf-8')).hexdigest()

if local_parq.exists():
    local_parq.unlink()

writer, total, t0 = None, 0, time.time()

for chunk in pd.read_csv(
    local_csv,
    chunksize=CHUNKSIZE,
    encoding='ascii',
    engine='python',
    encoding_errors='replace',
    sep=',',
):
    # Strip BOM + whitespace from column names
    chunk.columns = chunk.columns.astype(str).str.replace('\ufeff', '', regex=False).str.strip()

    # Cast numeric columns
    for col in FLOAT_COLS:
        if col in chunk.columns:
            chunk[col] = pd.to_numeric(chunk[col], errors='coerce').astype('float64')
    for col in INT_COLS:
        if col in chunk.columns:
            chunk[col] = pd.to_numeric(chunk[col], errors='coerce').astype('Int64')

    # Parse EFF_DATE if present
    if 'EFF_DATE' in chunk.columns:
        chunk['EFF_DATE'] = pd.to_datetime(chunk['EFF_DATE'], format='%d%b%Y', errors='coerce').dt.date

    # Hash PII
    for col in HASH_COLUMNS:
        if col in chunk.columns:
            chunk[col] = chunk[col].apply(hash_value)

    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(local_parq, table.schema, compression=COMPRESSION)
    writer.write_table(table)

    total += len(chunk)
    print(f'  ... {total:,} rows', end='\r')

if writer is not None:
    writer.close()

size_mb = local_parq.stat().st_size / 1e6
print(f'\n✓ {total:,} rows → {local_parq.name}  ({size_mb:.1f} MB in {time.time()-t0:.1f}s)')

## Cell 7 — Preview the Parquet (sanity check before upload)

Confirms hashing worked and the schema looks right.

In [ ]:
preview = pd.read_parquet(local_parq).head(5)
print(f'Columns ({len(preview.columns)}): {list(preview.columns)}')
print()
for col in HASH_COLUMNS:
    if col in preview.columns:
        sample = preview[col].dropna().iloc[0] if preview[col].notna().any() else '(all null)'
        looks_hashed = isinstance(sample, str) and len(sample) == 64
        mark = '✓' if looks_hashed else '⚠'
        print(f'  {mark} {col}: {str(sample)[:80]}')
print()
preview

## Cell 8 — Upload to GCS

Uses your `gcloud auth application-default login` credentials. If you haven't done that once on this machine, run it in Terminal: `gcloud auth application-default login`

In [ ]:
from google.cloud import storage
import time

print(f'Uploading {local_parq.name} → gs://{BUCKET}/{local_parq.name} ...')
client = storage.Client(project=GCP_PROJECT)
bucket_obj = client.bucket(BUCKET)
blob = bucket_obj.blob(local_parq.name)

t0 = time.time()
blob.upload_from_filename(str(local_parq))
size_mb = local_parq.stat().st_size / 1e6
print(f'✓ Uploaded {size_mb:.1f} MB in {time.time()-t0:.1f}s')
print(f'  URI: gs://{BUCKET}/{local_parq.name}')

## Cell 9 — Tidy up (optional)

Removes the local CSV (which can be large). The parquet is kept in `./out/`.

In [ ]:
KEEP_CSV = False   # set to True if you want to keep the raw CSV

if not KEEP_CSV and local_csv.exists():
    local_csv.unlink()
    print(f'✓ Removed local CSV: {local_csv.name}')
else:
    print(f'  Keeping local CSV: {local_csv}')

print(f'  Parquet retained:  {local_parq}')
print('\n✓ Done.')